<a href="https://colab.research.google.com/github/ylistokin/UNGA/blob/main/bootstrap_UN_totaldiff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#This file attempts to bootstrap the UN fixed effects explained estimates.

In [2]:
# Step 1: Mount Google Drive and key functions
from google.colab import drive
drive.mount('/content/gdrive')
!pip install linearmodels --quiet
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
from sklearn.utils import resample
from linearmodels.panel import RandomEffects
from tqdm import tqdm

Mounted at /content/gdrive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.7/115.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 3.3 MB/s eta 0:00:00


In [3]:
# Load your panel dataset
df = pd.read_stata("/content/gdrive/MyDrive/UNGA/final_data.dta")

# Set correct panel index
df = df.set_index(['country_num', 'year'])
df = df.sort_index()
#Creating a dataframe that only includes countries with multiple observations of positive share
df_positive = df[df['share'] > 0].copy()
df_positive = df_positive.groupby(level=0).filter(lambda g: len(g) >= 2)

#Data without Israel for decomposition estimation
df_ref = df[df['Israel_dummy'] == 0]
df_ref_pos = df_positive[df_positive['Israel_dummy'] == 0]


<ipython-input-3-2242353397>:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_positive = df_positive.groupby(level=0).filter(lambda g: len(g) >= 2)


In [4]:
#Calculating analytic estimates


In [5]:
# Fit fixed effects model in next three boxes
model_fe = PanelOLS.from_formula('share ~ battle_fat+TerritoryOccupied+human_rights_score+ western_affiliate+ NuclearPowerOutsideNPT+log_gdppc+log_population+NuclearPowerOutsideNPT+EntityEffects+TimeEffects', data=df_ref)
results_fe = model_fe.fit(cov_type='clustered', cluster_entity=True)
print(results_fe.summary)

model_re =RandomEffects.from_formula('share ~ battle_fat+TerritoryOccupied+human_rights_score+ western_affiliate+log_gdppc+log_population+NuclearPowerOutsideNPT+log_gdppc+log_population+NuclearPowerOutsideNPT+TimeEffects', data=df_ref)
results_re = model_re.fit(cov_type='clustered', cluster_entity=True)
print(results_re.summary)

model_fe_pos = PanelOLS.from_formula('share ~ battle_fat+TerritoryOccupied+human_rights_score+ western_affiliate+ NuclearPowerOutsideNPT+log_gdppc+log_population+NuclearPowerOutsideNPT+EntityEffects+TimeEffects', data=df_ref_pos)
results_fe_pos = model_fe_pos.fit(cov_type='clustered', cluster_entity=True)
print(results_fe_pos.summary)


# Get Group A coefficients (drop intercept and fixed effects)
beta_A_fe = results_fe.params[['battle_fat', 'TerritoryOccupied', 'human_rights_score', 'western_affiliate', 'NuclearPowerOutsideNPT', 'log_gdppc', 'log_population']]
beta_A_re = results_re.params[['battle_fat', 'TerritoryOccupied', 'human_rights_score', 'western_affiliate', 'NuclearPowerOutsideNPT', 'log_gdppc', 'log_population']]
beta_A_fe_pos = results_fe_pos.params[['battle_fat', 'TerritoryOccupied', 'human_rights_score', 'western_affiliate', 'NuclearPowerOutsideNPT', 'log_gdppc', 'log_population']]

# Get Group covariate means
mean_X_B = df[df['Israel_dummy'] == 1][['battle_fat', 'TerritoryOccupied', 'human_rights_score', 'western_affiliate', 'NuclearPowerOutsideNPT', 'log_gdppc', 'log_population']].mean()
mean_Y_B = df[df['Israel_dummy'] == 1]['share'].mean()
mean_X_A = df[df['Israel_dummy'] == 0][['battle_fat', 'TerritoryOccupied', 'human_rights_score', 'western_affiliate', 'NuclearPowerOutsideNPT', 'log_gdppc', 'log_population']].mean()
mean_Y_A = df[df['Israel_dummy'] == 0]['share'].mean()


# Get Group covariate means for positive results only
mean_X_B_pos = df_positive[df_positive['Israel_dummy'] == 1][['battle_fat', 'TerritoryOccupied', 'human_rights_score', 'western_affiliate', 'NuclearPowerOutsideNPT', 'log_gdppc', 'log_population']].mean()
mean_Y_B_pos = df_positive[df_positive['Israel_dummy'] == 1]['share'].mean()
mean_X_A_pos = df_positive[df_positive['Israel_dummy'] == 0][['battle_fat', 'TerritoryOccupied', 'human_rights_score', 'western_affiliate', 'NuclearPowerOutsideNPT', 'log_gdppc', 'log_population']].mean()
mean_Y_A_pos = df_positive[df_positive['Israel_dummy'] == 0]['share'].mean()




/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:1260: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
/usr/local/lib/python3.11/dist-packages/linearmodels/panel/data.py:590: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_mu = self._frame.groupby(level=level).transform("mean")
/usr/local/lib/python3.11/dist-packages/linearmodels/panel/data.py:590: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_mu = self._frame.groupby(level=level).transform("mean")
/usr/local/l

                          PanelOLS Estimation Summary                           
Dep. Variable:                  share   R-squared:                        0.1892
Estimator:                   PanelOLS   R-squared (Between):             -70.875
No. Observations:                7721   R-squared (Within):               0.1738
Date:                Tue, Jun 17 2025   R-squared (Overall):             -25.492
Time:                        15:51:25   Log-likelihood                 2.246e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      249.29
Entities:                         192   P-value                           0.0000
Avg Obs:                       40.214   Distribution:                  F(7,7480)
Min Obs:                       5.0000                                           
Max Obs:                       43.000   F-statistic (robust):             2.0354
                            

/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
/usr/local/lib/python3.11/dist-packages/linearmodels/panel/data.py:599: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  weighted_sum: DataFrame = frame.groupby(level=level).transform("sum")
/usr/local/lib/python3.11/dist-packages/linearmodels/panel/data.py:601: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sum_weights: DataFrame = frame.groupby(level=level).transform("sum

                        RandomEffects Estimation Summary                        
Dep. Variable:                  share   R-squared:                        0.1704
Estimator:              RandomEffects   R-squared (Between):             -0.1342
No. Observations:                7721   R-squared (Within):               0.1777
Date:                Tue, Jun 17 2025   R-squared (Overall):              0.0569
Time:                        15:51:26   Log-likelihood                 2.227e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      226.39
Entities:                         192   P-value                           0.0000
Avg Obs:                       40.214   Distribution:                  F(7,7714)
Min Obs:                       5.0000                                           
Max Obs:                       43.000   F-statistic (robust):             3.2872
                            

/usr/local/lib/python3.11/dist-packages/linearmodels/panel/data.py:590: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_mu = self._frame.groupby(level=level).transform("mean")
/usr/local/lib/python3.11/dist-packages/linearmodels/panel/data.py:590: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_mu = self._frame.groupby(level=level).transform("mean")
/usr/local/lib/python3.11/dist-packages/linearmodels/panel/data.py:590: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to

In [6]:
from numpy import dot
from numpy.linalg import multi_dot

# Compute explained component
explained_fe = np.dot(mean_X_A - mean_X_B, beta_A_fe)
explained_re = np.dot(mean_X_A - mean_X_B, beta_A_re)
explained_fe_pos = np.dot(mean_X_A_pos - mean_X_B_pos, beta_A_fe_pos)

# Compute unexplained component
overall_diff = mean_Y_A - mean_Y_B
overall_diff_pos = mean_Y_A_pos - mean_Y_B_pos


print(f"Overall difference: {overall_diff:.3f}")
print(f"Overall difference (excludes observations with no share): {overall_diff_pos:.3f}")
print(f"Explained component fixed effect: {explained_fe:.3f}")
print(f"Explained component random effect: {explained_re:.3f}")
print(f"Explained component fe with only positive share: {explained_fe_pos:.3f}")




Overall difference: -0.408
Overall difference (excludes observations with no share): -0.368
Explained component fixed effect: -0.101
Explained component random effect: -0.099
Explained component fe with only positive share: -0.198


In [7]:
#bootstrapping explained component standard errors for fixed effect regression
def bootstrap_oaxaca_fe(df, Israel_dummy, specification, entity_col, time_col, n_boot):
    np.random.seed(0)

    # Split groups
    df_israel = df[df[Israel_dummy] == 1]
    df_other = df[df[Israel_dummy] ==0]
    other_ids = df_other.index.get_level_values(entity_col).unique()

    explained_vals = []
    unexplained_vals = []

    # Extract the independent variables from the specification string
    independent_vars_str = specification.split('~')[1].split('+EntityEffects')[0].strip()
    independent_vars = [var.strip() for var in independent_vars_str.split('+')]

    # Filter for numeric columns among independent variables
    numeric_independent_vars = [var for var in independent_vars if pd.api.types.is_numeric_dtype(df[var])]


    for _ in range(n_boot):
        # Resample group B units with replacement
        resampled_ids = np.random.choice(other_ids, size=len(other_ids), replace=True)
        df_resampled = pd.concat([df_other.xs(i, level=entity_col) for i in resampled_ids],
                                 keys=resampled_ids, names=[entity_col])

        # Combine with fixed Israel group
        df_boot = pd.concat([df_israel, df_resampled])

        # Re-index for PanelOLS
        df_boot = df_boot.sort_index()

        # Fit FE models separately
        df_A = df_boot[df_boot[Israel_dummy] == 0]
        df_B = df_boot[df_boot[Israel_dummy] == 1]

        # Fit fixed effects model
        try:
            model_A = PanelOLS.from_formula(specification, data=df_A, check_rank=False).fit()


            # Compute Oaxaca components
            # Select only the numeric independent variables for calculating means
            xbar_A = df_A[numeric_independent_vars].dropna().groupby(level=1).mean().mean()
            #print(xbar_A)
            xbar_B = df_israel[numeric_independent_vars].mean()
            #print(xbar_B)
            beta_A = model_A.params
            #print(beta_A)


            # Ensure same covariates with estimated coefficients
            common_vars = [var for var in beta_A.index if var in numeric_independent_vars]
            #print(common_vars)
            xbar_A = xbar_A[common_vars]
            #print(xbar_A)
            xbar_B = xbar_B[common_vars]
            #print(xbar_B)



            explained = (xbar_B - xbar_A).values @ beta_A[common_vars].values
            print(explained)

            explained_vals.append(explained)
        except Exception as e:
            continue


    return {
        'explained_se_fe': np.std(explained_vals),
        'explained_mean_fe': np.mean(explained_vals)
    }

In [8]:
result_fe = bootstrap_oaxaca_fe(
    df=df,
    Israel_dummy='Israel_dummy',
    specification='share ~ battle_fat+TerritoryOccupied+human_rights_score+ western_affiliate+ NuclearPowerOutsideNPT+log_gdppc+log_population+EntityEffects+TimeEffects',
    entity_col='country_num',
    time_col='year',
    n_boot=500  # or any number of bootstrap replications
)

print("Bootstrapped SE for explained component:", result_fe['explained_se_fe'])
print("Bootstrapped mean for explained component:", result_fe['explained_mean_fe'])

/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:1260: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


-0.0010796753125559399


/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:1260: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


0.34481106576535386


/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:1260: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


...

-0.012324766056495504


/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:1260: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


0.1694011580512054


/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:1260: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


0.009446765620201771
Bootstrapped SE for explained component: 0.10687859042435864
Bootstrapped mean for explained component: 0.11592851150424521


In [9]:
#Moving on to random effects

In [10]:
#bootstrapping explained component standard errors for fixed effect regression
def bootstrap_oaxaca_re(df, Israel_dummy, specification, entity_col, time_col, n_boot):
    np.random.seed(0)

    # Split groups
    df_israel = df[df[Israel_dummy] == 1]
    df_other = df[df[Israel_dummy] ==0]
    other_ids = df_other.index.get_level_values(entity_col).unique()

    explained_vals = []
    unexplained_vals = []

    # Extract the independent variables from the specification string
    independent_vars_str = specification.split('~')[1].split('+EntityEffects')[0].strip()
    independent_vars = [var.strip() for var in independent_vars_str.split('+')]

    # Filter for numeric columns among independent variables
    numeric_independent_vars = [var for var in independent_vars if pd.api.types.is_numeric_dtype(df[var])]


    for _ in range(n_boot):
        # Resample group B units with replacement
        resampled_ids = np.random.choice(other_ids, size=len(other_ids), replace=True)
        df_resampled = pd.concat([df_other.xs(i, level=entity_col) for i in resampled_ids],
                                 keys=resampled_ids, names=[entity_col])

        # Combine with fixed Israel group
        df_boot = pd.concat([df_israel, df_resampled])

        # Re-index for PanelOLS
        df_boot = df_boot.sort_index()

        # Fit FE models separately
        df_A = df_boot[df_boot[Israel_dummy] == 0]
        df_B = df_boot[df_boot[Israel_dummy] == 1]

        # Fit fixed effects model
        try:
            model_A = RandomEffects.from_formula(specification, data=df_A, check_rank=False).fit()


            # Compute Oaxaca components
            # Select only the numeric independent variables for calculating means
            xbar_A = df_A[numeric_independent_vars].dropna().groupby(level=1).mean().mean()
            #print(xbar_A)
            xbar_B = df_israel[numeric_independent_vars].mean()
            #print(xbar_B)
            beta_A = model_A.params
            #print(beta_A)


            # Ensure same covariates with estimated coefficients
            common_vars = [var for var in beta_A.index if var in numeric_independent_vars]
            #print(common_vars)
            xbar_A = xbar_A[common_vars]
            #print(xbar_A)
            xbar_B = xbar_B[common_vars]
            #print(xbar_B)



            explained = (xbar_B - xbar_A).values @ beta_A[common_vars].values
            print(explained)

            explained_vals.append(explained)
        except Exception as e:
            continue


    return {
        'explained_se_re': np.std(explained_vals),
        'explained_mean_re': np.mean(explained_vals)
    }

In [11]:
result_re = bootstrap_oaxaca_re(
    df=df,
    Israel_dummy='Israel_dummy',
    specification='share ~ battle_fat+TerritoryOccupied+human_rights_score+ western_affiliate+ NuclearPowerOutsideNPT+log_gdppc+log_population+EntityEffects+TimeEffects',
    entity_col='country_num',
    time_col='year',
    n_boot=500  # or any number of bootstrap replications
)

print("Bootstrapped SE for explained component:", result_re['explained_se_re'])
print("Bootstrapped mean for explained component:", result_re['explained_mean_re'])

/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


0.005563897156358035


/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


0.34851819107927223


/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


...

/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


-0.0032024643514844127


/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


0.16441795816260818
0.013307408931962124
Bootstrapped SE for explained component: 0.1059609621441596
Bootstrapped mean for explained component: 0.11162201844060525


/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [12]:
#bootstrapping explained component standard errors for fixed effect regression, using only positive shares to deal with zero inflation problem
def bootstrap_oaxaca_pos(df, Israel_dummy, specification, entity_col, time_col, n_boot):
    np.random.seed(0)

    # Split groups
    df_israel = df[df[Israel_dummy] == 1]
    df_other = df[df[Israel_dummy] ==0]
    other_ids = df_other.index.get_level_values(entity_col).unique()

    explained_vals = []
    unexplained_vals = []

    # Extract the independent variables from the specification string
    independent_vars_str = specification.split('~')[1].split('+EntityEffects')[0].strip()
    independent_vars = [var.strip() for var in independent_vars_str.split('+')]

    # Filter for numeric columns among independent variables
    numeric_independent_vars = [var for var in independent_vars if pd.api.types.is_numeric_dtype(df[var])]


    for _ in range(n_boot):
        # Resample group B units with replacement
        resampled_ids = np.random.choice(other_ids, size=len(other_ids), replace=True)
        df_resampled = pd.concat([df_other.xs(i, level=entity_col) for i in resampled_ids],
                                 keys=resampled_ids, names=[entity_col])

        # Combine with fixed Israel group
        df_boot = pd.concat([df_israel, df_resampled])

        # Re-index for PanelOLS
        df_boot = df_boot.sort_index()

        # Fit FE models separately
        df_A = df_boot[df_boot[Israel_dummy] == 0]
        df_B = df_boot[df_boot[Israel_dummy] == 1]

        # Fit fixed effects model
        try:
            model_A = RandomEffects.from_formula(specification, data=df_A, check_rank=False).fit()


            # Compute Oaxaca components
            # Select only the numeric independent variables for calculating means
            xbar_A = df_A[numeric_independent_vars].dropna().groupby(level=1).mean().mean()
            #print(xbar_A)
            xbar_B = df_israel[numeric_independent_vars].mean()
            #print(xbar_B)
            beta_A = model_A.params
            #print(beta_A)


            # Ensure same covariates with estimated coefficients
            common_vars = [var for var in beta_A.index if var in numeric_independent_vars]
            #print(common_vars)
            xbar_A = xbar_A[common_vars]
            #print(xbar_A)
            xbar_B = xbar_B[common_vars]
            #print(xbar_B)



            explained = (xbar_B - xbar_A).values @ beta_A[common_vars].values

            explained_vals.append(explained)
        except Exception as e:
            continue


    return {
        'explained_se_pos': np.std(explained_vals),
        'explained_mean_pos': np.mean(explained_vals)
    }

In [13]:
result_pos = bootstrap_oaxaca_pos(
    df=df_positive,
    Israel_dummy='Israel_dummy',
    specification='share ~ battle_fat+TerritoryOccupied+human_rights_score+ western_affiliate+ NuclearPowerOutsideNPT+log_gdppc+log_population+EntityEffects+TimeEffects',
    entity_col='country_num',
    time_col='year',
    n_boot=500  # or any number of bootstrap replications
)

print("Bootstrapped SE for explained component:", result_pos['explained_se_pos'])
print("Bootstrapped mean for explained component:", result_pos['explained_mean_pos'])

/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
/usr/local/lib/python3.1

Bootstrapped SE for explained component: 0.10002430624123183
Bootstrapped mean for explained component: 0.13703579664880533


/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
/usr/local/lib/python3.11/dist-packages/linearmodels/panel/model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [14]:
!pip install python-docx
from docx import Document

from docx import Document

# Create a new Word document
doc = Document()

# Add a title
doc.add_heading("Table 3: Explained Differences by Model Specification", level=1)


# Add a table with 3 rows and 3 columns
table = doc.add_table(rows=4, cols=4)
table.style = 'Table Grid'


# Fill in the table

# Header row
table.cell(0, 0).text = "Model Specification"
table.cell(0, 1).text = "Explained Difference"
table.cell(0, 2).text = "Standard Error"
table.cell(0, 3).text = "Overall Difference"

# Row 1: Explained Difference (Random Effects)
table.cell(1, 0).text = "Random Effects"
table.cell(1, 1).text = f"{explained_re:.3f}"
table.cell(1, 2).text = f"({result_re['explained_se_re']:.3f})"
table.cell(1, 3).text = f"{overall_diff:.3f}"

# Row 2: Explained Difference (Fixed Effects)
table.cell(2, 0).text = "Fixed Effects"
table.cell(2, 1).text = f"{explained_fe:.3f}"
table.cell(2, 2).text = f"({result_fe['explained_se_fe']:.3f})"
table.cell(2, 3).text = f"{overall_diff:.3f}"

# Row 3: Explained Difference (Excludes observations with no share)
table.cell(3, 0).text = "Random Effects (excludes observations with no share)"
table.cell(3, 1).text = f"{explained_fe_pos:.3f}*"
table.cell(3, 2).text = f"({result_pos['explained_se_pos']:.3f})"
table.cell(3, 3).text = f"{overall_diff_pos:.3f}"

# Optionally set column headers
hdr_cells = table.rows[0].cells
table.columns[0].cells[0].text = "Model Specification"
table.columns[1].cells[0].text = "Explained Difference"
table.columns[2].cells[0].text = "Standard Error"
table.columns[3].cells[0].text = "Overall Difference"

# Add a paragraph below the table with the note
doc.add_paragraph()
doc.add_paragraph("Notes: Bootstrapped standard errors. ***, **, * denote 99%, 95%, and 90% statistical significance, respectively.")


# Save the document
doc.save("explained_differences_table.docx")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.0 MB/s eta 0:00:00


In [15]:
print(doc)

In [16]:
print(result_pos['explained_se_pos'])
print(result_re['explained_mean_re'])

0.10002430624123183
0.11162201844060525
